# Idea

The main idea is to expect symbols of same token that was just decoded. In that case each token is a sequence of symbols of limited subset of alphabet, and we use extra values to find the token ending.

Decoding example:
* While $V > 0$:
    * if type of token is unknown:
        $r_i = V \mod 3; V := V / 3$
        * if $r_i = 0$ it's `empty`
        * if $r_i = 1$ it's `number`
        * if $r_i = 2$ it's `word`

    * if token is `empty`
        * Nothing to decode, and next token has unknown type
    * if token is `number`
        * $s_0 = V \mod 10; V := V / 10$ it's first digit
        * for rest j:
            * $s_j = V \mod 13; V := V / 13$
            * if $s_j = 0$ it's separator and next token has unknown type
            * if $s_j < 11$ than $s_j - 1$ is a digit
            * if $s_j = 11$ it's separator and next token is number
            * if $s_j = 12$ it's separator and next token is word
    * if token is `word`
        * $s_0 = V \mod 26; V := V / 26$ it's first letter
        * for rest j:
            * $s_j = V \mod 29; V := V / 29$
            * if $s_j = 0$ it's separator and next token has unknown type
            * if $s_j < 27$ than $s_j - 1$ is a letter
            * if $s_j = 27$ it's separator and next token is number
            * if $s_j = 28$ it's separator and next token is word

Finaly we have those constants:

In [4]:
vector_lengths = [64, 128, 192, 256]

separator_descriptors_count = 3
letter_alphabet_size = 26
word_alphabet_size = letter_alphabet_size + separator_descriptors_count
word_alphabet_value_max = word_alphabet_size - 1
digit_alphabet_size = 10
number_alphabet_size = digit_alphabet_size + separator_descriptors_count
number_alphabet_value_max = number_alphabet_size - 1

# Density

## Best case

The token that starts from a lot of separator at beginnig and one single-digit number at the end, but this is too optimistic and not indicates practical density.

Best case is reachable on single number or single word identifier.

In [ ]:
for vector_length in vector_lengths:
    vector_value = (1 << vector_length) - 1
    vector_value //= separator_descriptors_count
    word_value = vector_value // letter_alphabet_size
    number_value = vector_value // digit_alphabet_size
    letters_count = 1  # the first letter consumed above (// letter_alphabet_size)
    digits_count = 1   # the first digit consumed above (// digit_alphabet_size)
    while word_value >= word_alphabet_value_max or number_value >= number_alphabet_value_max:
        if word_value >= word_alphabet_value_max:
            word_value //= word_alphabet_size
            letters_count += 1
        if number_value >= number_alphabet_value_max:
            number_value //= number_alphabet_size
            digits_count += 1

    letters_density = vector_length / letters_count
    digits_density = vector_length / digits_count
    print(f"{vector_length}-bits vector contains {letters_count} letters with density {letters_density:.3f}, or {digits_count} digits with density {digits_density:.3f}")

### Worst case

Worst case happens when all tokens are single letter words.

In [6]:
for vector_length in vector_lengths:
    vector_value = (1 << vector_length) - 1
    vector_value //= separator_descriptors_count
    word_value = vector_value
    number_value = vector_value
    letters_count = 0
    digits_count = 0
    while word_value >= letter_alphabet_size:
        word_value //= letter_alphabet_size
        letters_count += 1
        if word_value >= word_alphabet_value_max:
            word_value //= word_alphabet_size

    while number_value >= digit_alphabet_size:
        number_value //= digit_alphabet_size
        digits_count += 1
        if number_value >= number_alphabet_value_max:
            number_value //= number_alphabet_size

    letters_density = vector_length / letters_count
    digits_density = vector_length / digits_count
    print(f"{vector_length}-bits vector contains {letters_count} letters with density {letters_density:.3f}, or {digits_count} digits with density {digits_density:.3f}")

64-bits vector contains 7 letters with density 9.143, or 9 digits with density 7.111
128-bits vector contains 13 letters with density 9.846, or 18 digits with density 7.111
192-bits vector contains 20 letters with density 9.600, or 27 digits with density 7.111
256-bits vector contains 27 letters with density 9.481, or 36 digits with density 7.111


### Average case

Average density over realistic identifiers (80% words, 20% numbers — matching the README's $D_a$ definition). Words dominate, so the value sits a bit above the best word density.

In [ ]:
import random

# Per the README definition, the average assumes a realistic identifier: 80% word tokens,
# 20% number tokens. Word length ~N(5.5, 2), number length ~N(2, 1).
# Cost model (value domain):
#   - first token: one type marker (factor 3);
#   - word body:  first letter (/26) then (L-1) continuations (/29);
#   - number body: first digit (/10) then (L-1) continuations (/13);
#   - moving to the next token consumes one continuation slot of the *current* token's
#     alphabet (29 for a word, 13 for a number) that carries "separator + next type".
def avg_realistic(vl: int, p_word: float = 0.8, trials: int = 40_000) -> float:
    random.seed(7)
    total = 0
    for _ in range(trials):
        rest, symbols = (1 << vl) - 1, 0
        rest //= separator_descriptors_count          # initial type marker
        while True:
            if random.random() < p_word:
                first, cont, length = letter_alphabet_size, word_alphabet_size, max(1, round(random.gauss(5.5, 2)))
            else:
                first, cont, length = digit_alphabet_size, number_alphabet_size, max(1, round(random.gauss(2, 1)))
            if rest < first:
                break
            rest //= first; symbols += 1
            stop = False
            for _ in range(length - 1):
                if rest < cont:
                    stop = True; break
                rest //= cont; symbols += 1
            if stop or rest < cont:
                break
            rest //= cont                              # separator + next-type slot
        total += symbols
    return vl / (total / trials)

for vector_length in vector_lengths:
    print(f"Vector length: {vector_length} bits | avg(realistic 80/20) = {avg_realistic(vector_length):.3f} bits/sym")
